# 04 · 효과·불확실성·배포 판단

**핵심 실행 12분** · 해결 확인율 상승과 지연 악화가 공존할 때 근거를 설명합니다. 이번 과제의 핵심 파일입니다.

준비: 저장소 터미널에서 `python -m pip install -r requirements/week12-online.txt` 실행.
VS Code 오른쪽 위 **Select Kernel → 저장소 .venv의 Python 3.11**을 선택합니다.
다른 Notebook의 변수나 실행 순서를 요구하지 않습니다. 이 파일의 셀은 위에서 아래로 실행하세요.

**출처 구분:** CSV는 교수 작성 합성 사용자 데이터입니다. FastAPI 요청은 실제 실습 서비스 실행이며 실제 고객 실험이 아닙니다.
05번의 Ollama만 실제 로컬 모델을 호출합니다. Docker·클라우드 계정·유료 API는 필수가 아닙니다.

## 코드 셀 실행 지도

| 코드 셀 | 실행 | 확인할 결과 |
|---:|---|---|
| 1 | 환경 준비 | Python 3.11과 설정 버전 |
| 2 | 과제 사례 불러오기 | A 해결 확인율 약 55 |
| 3 | 절대 차이와 상대 개선 | 절대 차이는 약 9 |
| 4 | 95% 구간 직접 확인 | 95% 구간은 약 [5 |
| 5 | 판단 조건의 순서 | REVIEW / HOLD / INVALID를 각각 확인합니다 |
| 6 | 표본·기간과 중간 결과 | WAIT / WAIT / HOLD입니다 |
| 7 | 작은 효과와 필요한 표본 · 심화 | 탐지할 효과를 줄일수록 표본 요구량이 크게 늘어납니다 |
| 8 | 매일 유의성 확인 후 종료 · 심화 | 한 번의 종료 시점 판정보다 매일 확인한 거짓 양성 비율이 높습니다 |
| 9 | 내 판단 근거 저장 | 결과 JSON이 reports 폴더에 저장됩니다 |

In [ ]:
from pathlib import Path
import sys, json, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week12/lab/analysis.py").exists()), None)
if ROOT is None:
    raise RuntimeError("저장소 폴더를 VS Code로 열고 실행하세요.")
if sys.version_info[:2] != (3, 11):
    raise RuntimeError("저장소 .venv의 Python 3.11 커널을 선택하세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, HTML
from week12.lab.experiment import DEFAULT, assign
from week12.lab.analysis import analyze
from week12.lab.simulate import load_units
print("준비 완료:", sys.version.split()[0], "실험 설정", DEFAULT.version)


## 코드 셀 2 · 과제 사례 불러오기

slow는 정상 A/B 배정이지만 B의 지연이 큰 사례입니다. 여기서 구한 수치를 과제에 그대로 사용할 수 있습니다.

In [ ]:
data = load_units("slow")
report = analyze(data)
display(pd.DataFrame(report["arms"]).T)
print(report["decision"], report["reasons"])

**확인과 해석:** A 해결 확인율 약 55.0%, B 약 64.0%, B p95 약 2868ms, 결론 HOLD입니다.

**직접 설명:** 해결 확인율이 약 9%p 높아져도 배포를 보류하는 이유는 무엇인가요?

## 코드 셀 3 · 절대 차이와 상대 개선

퍼센트포인트(%p)와 상대 비율(%)을 구분합니다. 같은 결과를 두 방식으로 표현할 수 있습니다.

In [ ]:
a,b = report["arms"]["A"],report["arms"]["B"]
absolute = b["rate"]-a["rate"]
relative = absolute/a["rate"]
print(f"절대 차이 {100*absolute:.2f}%p / 상대 개선 {100*relative:.2f}%")

**확인과 해석:** 절대 차이는 약 9.02%p, 상대 개선은 약 16.4%입니다.

**직접 설명:** 9%와 9%p를 구분하지 않으면 어떤 오해가 생기나요?

## 코드 셀 4 · 95% 구간 직접 확인

독립 사용자의 이진 지표에 Newcombe–Wilson 방법을 사용합니다. 확률 95%로 특정 참값이 있다는 뜻으로 해석하지 않습니다.

In [ ]:
from week12.lab.analysis import difference_interval
interval = difference_interval(a["resolved"],a["n"],b["resolved"],b["n"])
print({k:round(100*interval[k],2) for k in ["difference","low","high"]}, "%p")
assert interval["low"] > 0

**확인과 해석:** 95% 구간은 약 [5.10, 12.91]%p입니다.

**직접 설명:** 이 구간이 양수라는 사실만으로 지연 문제를 무시해도 될까요?

## 코드 셀 5 · 판단 조건의 순서

데이터 불량 → 해석 중단, 위험 → 보류, 계획 미달 → 대기, 효과 불확실 → 결론 유보, 통과 → 사람 검토입니다.

In [ ]:
results = {name: analyze(load_units(name)) for name in ["healthy","slow","srm"]}
display(pd.DataFrame([{"사례":name,"판단":r["decision"],"이유":" / ".join(r["reasons"])} for name,r in results.items()]))
print("healthy라도 REVIEW이지 자동 배포가 아닙니다.")

**확인과 해석:** REVIEW / HOLD / INVALID를 각각 확인합니다.

**직접 설명:** 11주차 HOLD·REVIEW에 12주차 WAIT·INVALID가 추가된 이유는 무엇인가요?

## 코드 셀 6 · 표본·기간과 중간 결과

효과가 보이기 시작했다는 이유로 종료일을 당기지 않습니다. 운영 장애로 중단하는 것과 성과 승리 선언을 구분합니다.

In [ ]:
valid = load_units("healthy")
print("2일 중간 결과:",analyze(valid,elapsed_days=2)["decision"])
print("관찰 창 미완료:",analyze(valid,outcomes_mature=False)["decision"])
print("지연 위험 조기 중단:",analyze(data,elapsed_days=2)["decision"])

**확인과 해석:** WAIT / WAIT / HOLD입니다.

**직접 설명:** 왜 성과 판단은 기다리지만 심한 오류나 지연은 바로 중단할 수 있나요?

## 코드 셀 7 · 작은 효과와 필요한 표본 · 심화

이진 지표·50:50 배정·양측 5%·검정력 80%의 근사 계획입니다. 요청 수가 아니라 독립 사용자 수입니다.

In [ ]:
from week12.lab.analysis import sample_size
display(pd.DataFrame([{"목표 절대 차이":f"{100*d:.1f}%p","필요 사용자/버전":sample_size(.55,d)} for d in [.10,.05,.025]]))

**확인과 해석:** 탐지할 효과를 줄일수록 표본 요구량이 크게 늘어납니다.

**직접 설명:** 사용자 20명으로 작지만 유용한 효과를 확정하기 어려운 이유는 무엇인가요?

## 코드 셀 8 · 매일 유의성 확인 후 종료 · 심화

실제 효과가 없는 가상 세계를 반복합니다. 매일의 일반 p-value를 보고 유의하면 멈추는 규칙은 고정 시점 보장을 유지하지 않습니다.

In [ ]:
from week12.lab.analysis import peeking_demo
peek = peeking_demo()
display(pd.Series(peek))

**확인과 해석:** 한 번의 종료 시점 판정보다 매일 확인한 거짓 양성 비율이 높습니다. 값은 시뮬레이션의 seed와 반복 수에 종속됩니다.

**직접 설명:** 순차 검정은 매일 일반 검정을 반복하는 것과 무엇이 달라야 할까요?

## 코드 셀 9 · 내 판단 근거 저장

숫자 하나보다 결론·근거·다음 행동을 남깁니다. 과제는 한 페이지면 충분합니다.

In [ ]:
out = ROOT/"week12/lab/reports"
out.mkdir(exist_ok=True,parents=True)
(out/"my-slow-result.json").write_text(json.dumps(report,ensure_ascii=False,indent=2))
decision_note = {"결론":"HOLD", "주요 근거":"B 해결 확인율 상승, p95 1800ms 기준 위반", "다음 변경":"출력 길이를 줄이고 오프라인 회귀 평가 후 새 실험", "한계":"합성 데이터이며 실제 제품 성과가 아님"}
display(pd.Series(decision_note))
print(out/"my-slow-result.json")

**확인과 해석:** 결과 JSON이 reports 폴더에 저장됩니다. 판단 문장은 자신의 말로 보완하세요.

**직접 설명:** 답변 길이를 줄인 다음 11주차의 어떤 검사를 다시 실행해야 할까요?

## 복습과 다음 단계

결과를 맞추기 위해 p95 기준을 3000ms로 바꾸지 마세요. 변경안의 품질·속도를 다시 확인한 뒤 새 실험으로 검증합니다. 9–12주차 과제는 02번의 요청 기록 한 장과 이 파일의 slow 결과로 완성할 수 있습니다.

실행 오류는 Kernel을 다시 시작한 뒤 위에서부터 실행해 확인합니다. `ModuleNotFoundError`는 설치 터미널과 Notebook 커널이 같은 `.venv`인지 확인하세요.
실험 결과는 `week12/lab/reports/`에 저장하고 공개 GitHub에 학생 답안·개인정보를 올리지 않습니다.
[실행 안내](../WALKTHROUGH.md) · [100분 강의안](https://smilesjcha.github.io/llmops-16week/week12/lecture/) · [9–12주차 과제](https://smilesjcha.github.io/llmops-16week/week12/assignment/)
